# How well do traditional, expectations-augmented, and vacancies-based Phillips Curves explain US inflation across different macroeconomic regimes from 1958 to 2026?

### Myra Rafiq, Muhammad Abdullah, Saudha Shaheen






## Project Overview

The Phillips Curve in macroeconomics posits that when unemployment goes up, inflation tends to go down. This inverse relation was almost universally accepted in the 1950s. Shortly after, stagflation (high inflation, high unemployment) occured in the US in 1960s and 1970s, which meant the model had broken down.

From there, macroeconomists introduced the notion of expectations-augmented phillips curve. In the 1970s, after the Oil Crises, inflation expectations had become de-anchored, as people expected inflation to be persistent. High inflation the previous year was expected to be followed by high inflation the following year, regardless of Central Bank policies.

This project first visualizes the relations under the traditional Phillips Curve and the Expectations-Augmented Phillips Curve.

We then use a new measure, the vacancies per unemployed person(V/U), to understand how core inflation varies as the labor market becomes tighter.

This is a purely descriptive analysis. We are looking for patterns in the data rather than claiming that labor-market tightness causes inflation.

## 1. Data Collection

We use five monthly FRED series covering core inflation, headline inflation, unemployment, job openings, and number of unemployed people.

In [ ]:
import os
from fredapi import Fred

# Importing the libraries we need
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Connect to FRED
fred_api_key = os.getenv("FRED_API_KEY")
if not fred_api_key:
    raise ValueError("Set the FRED_API_KEY environment variable before running this notebook.")
fred = Fred(api_key=fred_api_key)

# Pull the six FRED series
unemployment_rate = fred.get_series('UNRATE')
number_unemployed = fred.get_series('UNEMPLOY')
job_openings = fred.get_series('JTSJOL')

# We import CPI data as a percentage change from last year so it is a measure of inflation.
core_inflation = fred.get_series('CPILFESL', units = 'pc1')
headline_inflation = fred.get_series('CPIAUCSL', units = 'pc1')




In [ ]:
unemployment_rate

The unit of observation is unemployment rate for a single month. Data is from Jan 1948 to September 2026.

In [ ]:
core_inflation

- The unit of observation is the inflation rate for a specific month. Inflation is calculated as the percentage change in the CPI level from the same month one year earlier. We use core inflation only as food and energy prices are very volatile.

- The first 12 observations return NaN because the dataset begins in January 1957 and there is no CPI observation from the previous year. Therefore, the calculated inflation series begins in January 1958 and extends through August 2026.

- The CPI uses 1982–1984 as its base period, with an index value of 100. However, this base does not affect the calculated inflation rate because inflation is based on the percentage change in CPI over the previous 12 months. For example, an inflation rate of 2.44616% in August 2026 means that the CPI was 2.44616% higher than in August 2025.


In [ ]:
number_unemployed

The unit of observation is the unemployment level for a specific month. It measures the aggregate number of people currently unemployed in the U.S., reported in thousands of persons. For example, a value of 7,109 in September 2026 represents 7,109,000 unemployed people in the U.S. in that month. Data is from January 1948 to September 2026.


In [ ]:
job_openings

The unit of observation are the number of job openings for a specific month. Data is from December 2000 to August 2026. The level is in thousands, so 7,079 in August 2026 means there were 7,079,000 job openings in the US in that month.




In [ ]:
headline_inflation

The unit of observation is headline inflation per month.

## 2. Combining the Data

Each FRED series is returned separately, so we combine the five series into one DataFrame using their monthly dates.

In [ ]:
# Combine all six series using their dates. Renamed as intial series had no names.
df = pd.concat([unemployment_rate.rename("unemployment_rate"),
                core_inflation.rename("core_inflation"),
                number_unemployed.rename("number_unemployed"),
                job_openings.rename("job_openings"),
               headline_inflation.rename("headline_inflation")],
               axis =1)

#dropping all of 1947 records:
df = df[df.index.year != 1947]
df

### Quick Look at the Data

In [ ]:
# View the first few observations
df.head()

In [ ]:
# View the last few observations
df.tail()

* Initial missing values because expected_inflation observations start in January 1982, core_inflation in January 1958, and job_openings in December 2000.

* Mising observations in September 2026 as core_inflation and job_openings observations end in August 2026.







## 3. Preliminary Data Check

We now check the size, structure, and completeness of the combined dataset.

In [ ]:
# Check the number of rows and columns
df.shape

We have 945 monthly observations and 5 columns.

### Column Names and Data Types

In [ ]:
# Check column names, data types, and non-missing values
df.info()

All columns follow the float64 datatype, which is expected as they are rates and values of economic indicators.

### Missing Values

We check for missing observations before choosing our analysis period.

In [ ]:
# Count missing values in each column
df.isna().sum()

The series do not all cover the same time period. This is because unemployment_rate begins in January 1948, expected_inflation in January 1982, core_inflation in January 1958, number_unemployed in January 1948, and job_openings in December 2000.  We will handle missing values according to the needs of our specific visualizations.

## 4. Creating the Analysis Variables

We now create a measure of labor-market tightness using job openings per unemployed person.  

### Labor-Market Tightness

We calculate job openings per unemployed person (V/U) as a measure of labor-market tightness. It measures for each unemployed person, the number of job openings available.

In [ ]:
# Calculate job openings per unemployed person
df["vu"] = df["job_openings"] / df["number_unemployed"]

# Since job_openings starts in December 2000, we take a look at the data beginning December 2000.
df.loc['2000-12-01': ,["job_openings", "number_unemployed", "vu"]].head()

### Tight and Slack Labor Markets

We classify months with more than one job opening per unemployed person as tight and the remaining months as slack.

In [ ]:
df["market_status"] = np.where(df["vu"] > 1, "tight", "slack")
df.loc['2000-12-01':, "market_status"].value_counts()

### Summary Statistics

We use basic summary statistics to understand the range and typical values of each variable.

The maximum unemployment_rate, maximum core_inflation, and maximum number_unemployed likely all happened during specific economic crises, which we will study under the visualizations. Since we will divide our dataset into eras, we will not drop the outliers as they inform our analysis and show the impact of economic events.

In [ ]:
# Summary statistics for the numerical variables
df.describe()

### Extreme Months and Checking for unusual observations

If one specific month particularly deviates from the trend, then we would flag it in case there is a measurement error. In our observations, as you can see, the top 5 values are all clustered together which means they were affected by economic events only.

In [ ]:
#  5 highest V/U months
#  V/U = number of job vacancies (V) divided by number of unemployed people (U)
#  A higher V/U means there are more job openings relative to unemployed workers

highest_vu = df.sort_values(
     "vu",
     ascending=False
 ).head(5)[["vu"]]

highest_vu

In [ ]:
# 5 lowest V/U months
# V/U = number of job vacancies (V) divided by number of unemployed people (U)
# A lower V/U means there are lesser job openings relative to unemployed workers

lowest_vu = df.sort_values(
    "vu"
).head(5)[["vu"]]
lowest_vu

In [ ]:
# 5 highest core inflation months
# Core inflation measures price changes of goods that are more volatile( excluding food and energy)

highest_core_inflation = df.sort_values(
     "core_inflation",
     ascending=False
).head(5)[["core_inflation"]]
highest_core_inflation

## 5. Visualization 1 - The traditional Phillips Curve relation (1958 -2026)


In this visualization, we plot unemployment_rate against core_inflation to check if there appears to be an inverse relation. We are not claiming any causal relation. We are just searching for a pattern.

Since we have core_inflation observations starting from January 1958, we begin our visualization from that point. This helps us have a wider span of data to look over (unlike in the instance we used job_openings, which had limited observations only from 2000 onwards)

In [ ]:
# Core Inflation observations are from January 1958 to August 2026 so we restrict the dataset for analysis.
df_1 = df.loc['1958-01-01': '2026-08-01', ['core_inflation', 'unemployment_rate']]
df_1

In [ ]:
# Checking for null values
df_1.isna().sum()

In [ ]:
# The missing value exists because of a 43 day federal government shutdown in October 2025
df_1[df_1.isna().any(axis =1)]

In [ ]:
# We drop the missing value
df_1 = df_1.dropna()
# Updated Shape and Info
print(df_1.shape)
print(df_1.info())

In [ ]:
fig,ax = plt.subplots(figsize =(12,6))
ax.plot(df_1.index, df_1["core_inflation"], color = "steelblue", label = "Core Inflation %")
ax.plot(df_1.index, df_1["unemployment_rate"], color = "orange", label = "Unemployment rate %")
ax.set_xlabel("Years")
ax.set_ylabel("Percent %")
ax.set_title("US Unemployment and Core Inflation 1958 - 2026")
ax.legend(loc = 'upper left')
ax.text(pd.to_datetime("1970-01-01"), 12, "Stagflation Era", color="red")
ax.text(pd.to_datetime("1980-01-01"), 2.7, "Volcker Shock", color="red")
ax.text(pd.to_datetime("2020-11-01"),9, "COVID-19", color="red");

* In the 1960s, we see that as core inflation remains low, unemployment is generally higher, which is in line with the inverse relation under Phillips curve.
* In the mid 1970s, both unemployment and inflation increase drastically, so there is no inverse relation. This is the stagflation (high unemployment, high inflation) era.
* In the 1980s, Chairman of Federal Reserve Paul Volcker in his 'war on inflation' massively restricted bank reserves to constrain demand and reduce inflation, which is why inflation falls from nearly 14% to around 3% in 1983.
* This leads to the period of 'Great Moderation' where the inflation remains somewhat stable. COVID-19 skyrockets the unemployment rate as firms cut workforces amidst the pandemic.



## 6. Visualization 2 - Expectations-Augmented Phillips Curve (1970 - 2026)

* When the traditional Phillips curve relation broke down in the 1970s,   macroeconomists believed it was due to de-anchoring of expectations. Before in the 1960s, people expected the inflation to be fixed. Because of oil crises and global economic changes in the 1970s, now people in the US did not believe that the Fed could keep prices stable, so they expected high prices to be persistent.

* Inflation in one year was equal to inflation in the previous year as high prices were expected. So, the phillips curve had evolved from a relationship between inflation and unemployment to now a relationship between inflation gap(inflation - expected inflation) and unemployment.

* We test this relationship from 1970 to 2026. This time, we use headline inflation to account for energy prices as well which were affected by supply shocks and global oil crises.


In [ ]:
# Restricting the dataset for analysis. We start at 1969 so we can get a value for 1970 as seen down below.
df_2 = df.loc["1969-01-01" : "2026-08-01", ["headline_inflation", "unemployment_rate"]]

In [ ]:
# We groupby years to get the average inflation each year.
df_2 = df_2.groupby(df_2.index.year).agg(
    headline_inflation_yearly = ("headline_inflation", "mean"),
    unemployment_yearly = ("unemployment_rate", "mean"))

In [ ]:
# Now we calculate the inflation gap, which is inflation minus inflation in the previous year.
df_2["headline_inflation_yearly_1m"] = df_2["headline_inflation_yearly"].shift(1)
df_2["inflation_gap"] = df_2["headline_inflation_yearly"] - df_2["headline_inflation_yearly_1m"]


In [ ]:
# Checking for null values
df_2.isna().sum()

In [ ]:
# Since we have no observations for 1968, we lose the observations for 1969. Hence, we drop null values
df_2 = df_2.dropna()
# Updated Shape and Info
print(df_2.shape)
print(df_2.info())

In [ ]:
# We introduce different eras to understand the shape of the Phillips Curve
df_2 = df_2.copy()
df_2["era"] = ""
df_2.loc[1970:1983, "era"] = "1970-1983 (Accelerationist)"
df_2.loc[1984:2007, "era"] = "1984-2007 (Great Moderation)"
df_2.loc[2008:2026, "era"] = "2008-2026 (Contemporary)"

In [ ]:
v2 = sns.relplot(data = df_2, x = "unemployment_yearly", y = "inflation_gap", col = "era", kind = "scatter", hue = "era", legend = False,
                 palette = "crest", s = 140)
v2.set_xlabels("Unemployment Rate (%)")
v2.set_ylabels("Inflation Gap (%)")
v2.set_titles(col_template="{col_name}")
v2.fig.suptitle("Expectations-Augmented Phillips Curve Across Eras", y = 1.1);


* In the 1970s, the expectations-augmented Phillips Curve follows an inverse relation. After we adjusted for inflation expectations (as people expected them to be based on previous year), we see an inverse relation like the traditional Phillips Curve.

* In the Great Moderation era, inflation expectations had become anchored i.e stable after the tight policies of Paul Volcker.

* In the modern times, the Great Recession and COVID led to spikes in inflation/deflation and unemployment. Largely, inflation expectations remained contained.

* Note, we do not claim that the invserse relation observed in the accelerationist portion is causal. It is just a relationship. There are many other variables that affect both inflation and unemployment which we have not accounted for.

## 7. Visualization 3 - Vacancies-based Phillips Curve

We now propose a new measure to check the relation between inflation and unempoyment. Instead of using the unemployment rate which only focuses on the demand side, we now use job openings (vacancies) / unemployed people (V/U). This allows us to incorporate the supply side as well to understand the overall labor market tightness.

In [ ]:
# Since we only have job openings data from Dec 2000, we restrict the dataset.
df_3 = df.loc["2000-12-01":"2026-08-01", ["unemployment_rate", "core_inflation", "vu"]]

In [ ]:
# We first create bin boundaries
bins = [0, 0.5, 0.75, 1, 1.25, 1.5, 2, 3]
labels = ["<0.5", "0.5-0.75", "0.75-1", "1-1.25", "1.25-1.5", "1.5-2", ">2"]

# Now we add the bucket column
df_3["vu_bucket"] = pd.cut(df_3["vu"], bins = bins, labels =labels)

In [ ]:
bucket_avg = (df_3.groupby("vu_bucket", observed=False)["core_inflation"].mean().reset_index())

ax = sns.barplot(data=bucket_avg, x="vu_bucket", y="core_inflation", hue="vu_bucket", palette="Reds", legend=False)
plt.xlabel("V/U Bucket")
plt.ylabel("Avg Core Inflation (%)")
plt.title("Core Inflation by V/U Bucket");
plt.show()

* As expected, when the labor market is tight i.e v/u is greater than 1, then average core inflation tends to rise. This makes intuitive sense because there is less unemployment in the economy. Hence, aggregate demand rises as consumption and investment increase in the economy.

## 8. Limitations and Next Steps

* (1.) We did not account for other variables that affect the tradeoff between unemployment and inflation. This is a purely exploratory analysis, supplemented with macroeconomic history.
* (2.) Our data is only based on the US. Whether the Phillips Curve is a universal phenomenon is an entirely different question in itself.
* (3.) In the second visaluzation, we assume that expectation is equal to the last years visualization. This is a generalization adapted from Olivier Blanchard's textbook Macroeconomics.

### Next Steps: Test the relationship in other countries. Use  a different model for expectations. Add more robustness checks.  